# 04. Normalizing Flows: RealNVP, Glow, and Spline Flows

- **Category:** 05_architectures / 05_generative_models
- **Lineage & Citations:** Laurent Dinh et al., 2014/2016 (*NICE*, *RealNVP*); Diederik P. Kingma & Prafulla Dhariwal, 2018 (*Glow*); Conor Durkan et al., 2019 (*Neural Spline Flows*).
- **Purpose:** Formulates exact-likelihood generative modeling via invertible bijective transformations, triangular Jacobian determinants in RealNVP, and invertible $1\times 1$ convolutions in Glow.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [GANs DCGAN WGAN GP and StyleGAN Concepts](03_gans_dcgan_wgan_gp_and_stylegan_concepts.ipynb) $\to$ `🔗 Downstream Syntheses:` [Diffusion DDPM DDIM EDM and DiT](05_diffusion_ddpm_ddim_edm_and_dit.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**
> - **Fitting Continuous Flows to Discrete Data Without Dequantization:** Fitting raw pixel integers $\{0, \dots, 255\}$ directly leads to degenerate infinite Dirac density spikes; uniform dequantization noise is strictly mandatory.
> - **Unconstrained Scale Network Outputs:** Without bounding $s(x_A)$ via $\tanh$, $\exp(s)$ overflows numerical floating point range and causes `NaN` explosions.

## Mathematical Foundations & Governing Equations

### 1. Change-of-Variables Theorem
$$\log p_x(x) = \log p_z(f(x)) + \log \left| \det \frac{\partial f(x)}{\partial x} \right|$$

### 2. RealNVP Affine Coupling
$$y_A = x_A, \quad y_B = x_B \odot \exp(s(x_A)) + t(x_A), \quad \log |\det \mathbf{J}| = \sum_i s(x_A)_i$$
Exact algebraic inverse without matrix inversion:
$$x_A = y_A, \quad x_B = (y_B - t(y_A)) \odot \exp(-s(y_A))$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_a = weights["x_a"]
x_b = weights["x_b"]
s_w = weights["s_w"]
s_b = weights["s_b"]
t_w = weights["t_w"]
t_b = weights["t_b"]
x_glow = weights["x_glow"]
w_glow = weights["w_glow"]
print(f"Loaded {len(weights)} golden tensors on {env.device}, x_a: {list(x_a.shape)}, w_glow: {list(w_glow.shape)}")

In [ ]:
def realnvp_forward(x_a, x_b, s_w, s_b, t_w, t_b):
    s = torch.tanh(x_a @ s_w + s_b)
    t = x_a @ t_w + t_b
    y_a = x_a
    y_b = x_b * torch.exp(s) + t
    log_det = s.sum(dim=-1)
    return y_a, y_b, log_det

def realnvp_inverse(y_a, y_b, s_w, s_b, t_w, t_b):
    s = torch.tanh(y_a @ s_w + s_b)
    t = y_a @ t_w + t_b
    x_a = y_a
    x_b = (y_b - t) * torch.exp(-s)
    log_det = -s.sum(dim=-1)
    return x_a, x_b, log_det

def glow_1x1_conv(x, w):
    y = x @ w
    _, logabsdet = torch.slogdet(w)
    log_det = logabsdet.unsqueeze(0).expand(x.shape[0])
    x_rec = y @ torch.linalg.inv(w)
    return y, log_det, x_rec

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
y_a, y_b, log_det = realnvp_forward(x_a, x_b, s_w, s_b, t_w, t_b)
err_ya = float(torch.max(torch.abs(y_a - weights["y_a"])))
err_yb = float(torch.max(torch.abs(y_b - weights["y_b"])))
err_ld = float(torch.max(torch.abs(log_det - weights["log_det"])))
assert err_ya < 1e-6, f"y_a error: {err_ya:.2e}"
assert err_yb < 1e-6, f"y_b error: {err_yb:.2e}"
assert err_ld < 1e-6, f"log_det error: {err_ld:.2e}"

x_a_rec, x_b_rec, _ = realnvp_inverse(y_a, y_b, s_w, s_b, t_w, t_b)
recon_err = float(torch.max(torch.abs(x_b - x_b_rec)))
assert recon_err < 1e-6, f"RealNVP exact roundtrip reconstruction error: {recon_err:.2e}"

y_g, ld_g, x_g_rec = glow_1x1_conv(x_glow, w_glow)
err_yg = float(torch.max(torch.abs(y_g - weights["y_glow"])))
assert err_yg < 1e-6, f"Glow 1x1 conv error: {err_yg:.2e}"
print(f"Parity Verified: y_b={err_yb:.2e}, log_det={err_ld:.2e}, RealNVP Recon={recon_err:.2e}, Glow={err_yg:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_realnvp = benchmark_fn(lambda: realnvp_forward(x_a, x_b, s_w, s_b, t_w, t_b), iters=50, name="RealNVP Forward")
b_glow = benchmark_fn(lambda: glow_1x1_conv(x_glow, w_glow), iters=50, name="Glow Invertible 1x1")
print(format_benchmark_table([b_realnvp, b_glow]))

# Visualization: Coupling Latent Space Invertible Transformation
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].scatter(x_a[:, 0].numpy(), x_b[:, 0].numpy(), color="royalblue", alpha=0.8, label="Base x")
ax[0].set_title("Input Data Distribution (x_A, x_B)")
ax[0].legend(fontsize=8)

ax[1].scatter(y_a[:, 0].numpy(), y_b[:, 0].numpy(), color="crimson", alpha=0.8, label="Transformed y")
ax[1].set_title("Transformed Flow Space (y_A, y_B)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse vs Corrected Implementation: Unbounded Scale Output Exploding
# Without tanh activation, unbounded s causes exp(s) to blow up into inf / NaN!
s_unbounded = x_a @ s_w * 50.0  # Simulated unconstrained scale network
exp_unbounded = torch.exp(s_unbounded)
overflow_count = int(torch.isinf(exp_unbounded).sum().item())
print(f"Unconstrained exp(s) overflow count (inf values): {overflow_count}")

s_bounded = torch.tanh(x_a @ s_w + s_b)
exp_bounded = torch.exp(s_bounded)
assert not torch.isnan(exp_bounded).any() and not torch.isinf(exp_bounded).any()
assert exp_bounded.max() <= math.e + 1e-3
print("Demonstrated requirement of bounded tanh activations in coupling scale networks.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Triangular Jacobians Enable Tractable Likelihoods:** RealNVP transforms half the channels while holding the other constant, achieving $O(D)$ log-determinants.
2. **Invertible Convolutions Generalize Permutations:** Glow's $1\times 1$ invertible convs learn channel mixings without volume distortion.
3. **Exact Inversion With Zero Gradient Descent:** Forward and inverse passes share the exact same conditioning subnets without matrix inversion.

---
### Cross-Domain Generalisation Matrix
- **Physics & Lattice Field Theory:** Invertible flows sample equilibrium thermodynamic distributions without autocorrelation slowing.
- **Real-Time Speech Synthesis:** WaveGlow synthesizes natural speech in single-pass parallel decoding.
- **Variational Bayesian Inference:** Normalizing flows transform simple posteriors into multi-modal complex distributions.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_normalizing_flows_realnvp_glow_and_spline_flows.md](../../../markdown/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.md)
- **PyTorch 2.x Module:** [04_normalizing_flows_realnvp_glow_and_spline_flows.py](../../../pytorch2/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.py)
- **Functional JAX Module:** [04_normalizing_flows_realnvp_glow_and_spline_flows.py](../../../jax/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.py)
- **Candle Rust Module:** [04_normalizing_flows_realnvp_glow_and_spline_flows.rs](../../../candle/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.rs)
- **Hardware Kernel:** [04_normalizing_flows_realnvp_glow_and_spline_flows.rs](../../../kernels/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.rs)
- **Universal Golden SafeTensors:** [04_normalizing_flows_realnvp_glow_and_spline_flows.safetensors](../../../weights/05_architectures/05_generative_models/04_normalizing_flows_realnvp_glow_and_spline_flows.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 03. GANs DCGAN WGAN GP and StyleGAN Concepts](03_gans_dcgan_wgan_gp_and_stylegan_concepts.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 05. Diffusion DDPM DDIM EDM and DiT](05_diffusion_ddpm_ddim_edm_and_dit.ipynb) &rarr;